In [1]:
import pandas as pd
import itertools
from itertools import combinations
from tqdm import tqdm 
import csv

In [140]:
# 1. Read data
df = pd.read_csv(
    "Food_Inspections_20260328.csv",
)

# Drop free-text and derived/duplicate columns
# DROP_COLS = ["Violations", "Location"]
# df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])

# # Drop rows with nulls (nulls create false FD violations)
# df = df.dropna()

print(f"  Full dataset : {len(df):,} rows, {len(df.columns)} columns")
print(f"  Columns      : {list(df.columns)}\n")

cols = list(df.columns)

  Full dataset : 307,883 rows, 17 columns
  Columns      : ['Inspection ID', 'DBA Name', 'AKA Name', 'License #', 'Facility Type', 'Risk', 'Address', 'City', 'State', 'Zip', 'Inspection Date', 'Inspection Type', 'Results', 'Violations', 'Latitude', 'Longitude', 'Location']



# 0 - Risk

## 1 - Value counts

In [141]:
df['Risk'].value_counts()

Risk
Risk 1 (High)      228444
Risk 2 (Medium)     55262
Risk 3 (Low)        24005
All                    85
Name: count, dtype: int64

## 2 - Check "all"

In [142]:
filtered_df = df[df['Risk']=="All"]
filtered_df

,Inspection ID,DBA Name,AKA Name,License #,Facility Type,Risk,Address,City,State,Zip,Inspection Date,Inspection Type,Results,Violations,Latitude,Longitude,Location
31,2633024,PAKS ON THE PIER,PAKS ON THE PIER,3077934.0,NaN,All,600 E GRAND AVE,CHICAGO,IL,60611.0,03/24/2026,License,No Entry,NaN,41.892094,-87.611570,"(41.892094136861786, -87.61156988394656)"
435,2632484,ARLA,ARLA,3073930.0,Restaurant,All,15 E OAK ST,CHICAGO,IL,60611.0,03/12/2026,License,Not Ready,NaN,41.900647,-87.627547,"(41.9006472539258, -87.62754741411463)"
717,2632127,MOLINO LOS HERMANOS,MOLINO LOS HERMANOS,3073490.0,NaN,All,4666-4668 N BROADWAY,CHICAGO,IL,60640.0,03/04/2026,License,Not Ready,NaN,41.967086,-87.658715,"(41.96708593016735, -87.65871498931575)"
800,2632036,33 IROBOT CAFE,NaN,3073505.0,NaN,All,2101 S STATE ST STE 101-102,CHICAGO,IL,60616.0,03/03/2026,License,Not Ready,NaN,41.854184,-87.626965,"(41.854184344584326, -87.6269648513787)"
906,2631971,TOPOLI PERSIAN & MEDITERRANEAN CUISINE CORPORA...,TOPOLI PERSIAN & MEDITERRANEAN CUISINE,3073427.0,NaN,All,5335 N SHERIDAN RD,CHICAGO,IL,60640.0,02/27/2026,License,Not Ready,NaN,41.979185,-87.654912,"(41.97918539940176, -87.65491162783847)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
248788,1084429,SHAY BOO'S KITCHEN,NaN,2240189.0,NaN,All,214 N HOMAN AVE,CHICAGO,IL,60624.0,03/07/2013,OWNER SUSPENDED OPERATION/LICENSE,Fail,NaN,41.885103,-87.711278,"(41.88510313446372, -87.71127828682762)"
252446,1319296,SHARKS,NaN,0.0,NaN,All,660 E 79TH ST,CHICAGO,IL,60619.0,12/20/2012,Complaint,Business Not Located,NaN,41.751352,-87.607722,"(41.75135220978944, -87.60772162070809)"
264617,1214783,CHOI'S CHINESE STEAK HOUSE,CHOI'S CHINESE STEAK HOUSE,1095422.0,NaN,All,2638 N MILWAUKEE AVE,CHICAGO,IL,60647.0,05/10/2012,Canvass,Out of Business,NaN,41.929670,-87.708624,"(41.929670220351134, -87.70862407382609)"
269981,660163,TIKIM RESTAURANT,TIKIM RESTAURANT,1356156.0,NaN,All,3116 W LAWRENCE AVE,CHICAGO,IL,60625.0,01/25/2012,Canvass,Out of Business,NaN,41.968579,-87.706774,"(41.96857895603585, -87.7067738901748)"


## 3 - Drop all "all"?

In [143]:
df = df[df['Risk'] != 'All']

# 1 - Zip

## 1 - Value counts

In [144]:
df['Zip'].value_counts()

Zip
60614.0    11444
60647.0    11341
60657.0    10461
60611.0     9700
60618.0     9667
           ...  
60056.0        1
60044.0        1
60108.0        1
60478.0        1
60148.0        1
Name: count, Length: 132, dtype: int64

## 2 - Keep only Chicago zips

In [145]:
import numpy as np

# 1. Format the Zip column properly (convert to string, drop the '.0')
df['Zip'] = df['Zip'].astype(str).str.replace('.0', '', regex=False)
df['Zip'] = df['Zip'].replace('nan', np.nan)

# 2. The Hard Filter: Keep ONLY rows with a Chicago zip code prefix
chicago_prefixes = ('606', '607', '608')
df = df[df['Zip'].str.startswith(chicago_prefixes, na=False)].reset_index(drop=True)

# 2 - State

## 1 - Value counts

In [146]:
df['State'].value_counts()

State
IL    307467
Name: count, dtype: int64

## 2 - Check data beyond Illinois

In [147]:
targets = ['IN', 'CA', 'WI', 'CO', 'NY']
filtered_df = df[df['State'].isin(targets)]
filtered_df

,Inspection ID,DBA Name,AKA Name,License #,Facility Type,Risk,Address,City,State,Zip,Inspection Date,Inspection Type,Results,Violations,Latitude,Longitude,Location


## 3 - Drop all outside of Illinois

In [148]:
df = df[df['State'] == 'IL']

In [149]:
# state column will be useless, as it will all be illinois

# 3 - City

## 1 - Value counts

In [150]:
df['City'].value_counts()

City
CHICAGO                306515
Chicago                   499
chicago                   161
CCHICAGO                   62
CHicago                    22
CHICAGOCHICAGO             12
CHICAGOO                   11
CICERO                      9
CHICAGO.                    8
INACTIVE                    8
NILES NILES                 7
312CHICAGO                  7
CHCHICAGO                   6
CHARLES A HAYES             4
ALSIP                       3
CHCICAGO                    3
CHICAGOI                    3
CH                          2
EVERGREEN PARK              2
BURNHAM                     2
CHICAGOC                    2
chicagoBEDFORD PARK         1
Norridge                    1
alsip                       1
Name: count, dtype: int64

## 2 - Correct Chicago typos

In [151]:
# Force uppercase and strip hidden spaces
df['City'] = df['City'].str.upper().str.strip()

# Define your mapping dictionary for the typos you found
city_corrections = {
    'CCHICAGO': 'CHICAGO',
    'CHICAGOCHICAGO': 'CHICAGO',
    'CHICAGOO': 'CHICAGO',
    'CHICAGO.': 'CHICAGO',
    'CHCHICAGO': 'CHICAGO',
    'CHICAGOI': 'CHICAGO',
    'CHCICAGO': 'CHICAGO',
    'CHICAGOC': 'CHICAGO',
    '312CHICAGO': 'CHICAGO',
    'CH': 'CHICAGO',
    'CHARLES A HAYES': 'CHICAGO', # building is in Chicago
    'INACTIVE': 'CHICAGO', # only refer to 1 place in Chicago
    
}

# Apply the corrections to the column
df['City'] = df['City'].replace(city_corrections)

## 3 - Fill NA City values with Chicago after confirming they are all Chicago

In [152]:
df[df['State'].isna()][['City', 'Zip', 'Latitude', 'Longitude']].head(43)

mask = (
    df['City'].isna() &
    df['Zip'].notna() &
    (df['Zip'].astype(str).str.startswith('606'))
)

df.loc[mask, 'City'] = 'CHICAGO'

## 3 - Check

In [153]:
df['City'].value_counts()

City
CHICAGO                307441
CICERO                      9
NILES NILES                 7
ALSIP                       4
EVERGREEN PARK              2
BURNHAM                     2
CHICAGOBEDFORD PARK         1
NORRIDGE                    1
Name: count, dtype: int64

## 4 - Handle out of Chicago and Illinois data

### Option 1: Drop all data beyond Chicago

In [154]:
# The final purge: Keep only the exact match for Chicago
# df = df[df['City'] == 'CHICAGO'].reset_index(drop=True)

In [155]:
# city column will be useless, as it will all be chicago

### Option 2: Keep suburbs 

In [156]:
# Dictionary to fix the suburban and out-of-state typos
suburb_corrections = {
    'NILES NILES': 'NILES',
    'CHICAGOBEDFORD PARK': 'BEDFORD PARK',
}

# Apply the corrections to the City column
df['City'] = df['City'].replace(suburb_corrections)

In [157]:
# Cicero, Alsip, Evergreen Park, Norridge = chicago suburbs
# Chicagobedford park, niles niles 

In [158]:
df['City'].value_counts()

City
CHICAGO           307441
CICERO                 9
NILES                  7
ALSIP                  4
EVERGREEN PARK         2
BURNHAM                2
BEDFORD PARK           1
NORRIDGE               1
Name: count, dtype: int64

# 5 - Address

## 1 - Value counts

In [159]:
df['Address'].value_counts()

Address
11601 W TOUHY AVE      2807
11601 W TOUHY AVE      1061
5700 S CICERO AVE       771
2300 S THROOP ST        596
500 W MADISON ST        532
                       ... 
7832 S western AVE        1
4001 N BROADWAY           1
2314 W 95TH ST            1
4041 W NORTH AVE          1
4925 N Broadway           1
Name: count, Length: 32885, dtype: int64

In [160]:
# Force everything to uppercase and strip out all leading/trailing hidden spaces
df['Address'] = df['Address'].str.upper().str.strip()

# Check your work - those two Touhy rows should instantly merge into one!
print(df['Address'].value_counts())

Address
11601 W TOUHY AVE        3875
5700 S CICERO AVE         884
2300 S THROOP ST          760
500 W MADISON ST          621
131 N CLINTON ST          578
                         ... 
707 E 37TH ST (3700S)       1
2 E 112 TH ST               1
2555 N ELSTON AVE           1
739 W 129TH PL              1
5600 N RIDGE BLDG           1
Name: count, Length: 20021, dtype: int64
